# Practical example. Audiobooks

**Preprocess the data. Balance the dataset. Create 3 datasets: training, validation, and test. Save the newly created sets in a tensor friendly format (e.g. filename.npz)**

Since we are dealing with real life data, we will need to preprocess it a bit. This is the relevant code, which is not that hard, but refers to data engineering more than machine learning.

If you want to know how to do that, go through the code and the comments. In any case, this should do the trick for all datasets organized in the way: many inputs, and then 1 cell containing the targets (all supervised learning datasets).

Note that we have removed the header row, which contains the names of the categories. We simply want the data.

## Extracting the data from csv

In [18]:
import numpy as np
from sklearn import preprocessing

raw_csv_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')

unscaled_inputs_all = raw_csv_data[:,1:-1] #excluding the first and last column
targets_all = raw_csv_data[:,-1] #last target column of the csv

## Balancing the dataset

In [19]:
# we will count the number of 0s and 1s and keep as many 0s and 1s possible rest will be deleted
num_one_targets = int(np.sum(targets_all))

zero_targets_counter = 0 # Counter to keep track of how many 0s we've seen so far
indices_to_remove = [] # List to store the row indices of extra 0s that we'll remove

# Loop through every target in the dataset
for i in range(targets_all.shape[0]):
    if targets_all[i]== 0:
        zero_targets_counter += 1 # Increment the counter each time we encounter a 0
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i) # Once we've already kept enough 0s (equal to the number of 1s),mark any further 0s for removal

# Remove the excess 0-target rows from the inputs,
# leaving a roughly balanced number of 0s and 1s
unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
#Remove the same excess rows from the targets, so inputs and targets stay aligned
targets_equal_priors = np.delete(targets_all,indices_to_remove, axis= 0) 

## Standardize the inputs

In [20]:
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

## Shuffle the data 

In [21]:
# Create an array of indices, one for each row in the scaled inputs
# (e.g., if there are 1000 rows, this gives [0, 1, 2, ..., 999])
shuffled_indices = np.arange(scaled_inputs.shape[0])
# Shuffle those indices randomly in place —
np.random.shuffle(shuffled_indices)

# Use the shuffled indices to reorder the inputs...
shuffled_inputs = scaled_inputs[shuffled_indices]
# ...and apply the *same* shuffled order to the targets,
shuffled_targets = targets_equal_priors[shuffled_indices]

## Split dataset into train, validation, test

In [22]:
# Total number of samples in the (already shuffled) dataset
samples_count = shuffled_inputs.shape[0]

# Split sizes: 80% train, 10% validation, 10% test
train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
# Test gets whatever is left, so all samples are accounted for
test_samples_count = samples_count - train_samples_count - validation_samples_count

#Training set: first chunk of the shuffled data 
train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

#Validation set: next chunk, right after the training data ---
validation_inputs = shuffled_inputs[train_samples_count:train_samples_count + validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count + validation_samples_count]

#Test set: whatever remains after train + validation ---
test_inputs = shuffled_inputs[train_samples_count + validation_samples_count:]
test_targets = shuffled_targets[train_samples_count + validation_samples_count:]

# Sanity check: for each set, print the number of 1s, total sample count,
# and the proportion of 1s — this confirms the classes are still balanced
# (~50/50) across train/validation/test after shuffling and splitting
print(np.sum(train_targets), train_samples_count, np.sum(train_targets) / train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets) / test_samples_count)

1787.0 3579 0.4993014808605756
220.0 447 0.49217002237136465
230.0 448 0.5133928571428571


## save the dataset in *.npz

In [23]:
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

# Practical example. Audiobooks



## Problem



You are given data from an Audiobook app. Logically, it relates only to the audio versions of books. Each customer in the database has made a purchase at least once, that's why he/she is in the database. We want to create a machine learning algorithm based on our available data that can predict if a customer will buy again from the Audiobook company.

The main idea is that if a customer has a low probability of coming back, there is no reason to spend any money on advertizing to him/her. If we can focus our efforts ONLY on customers that are likely to convert again, we can make great savings. Moreover, this model can identify the most important metrics for a customer to come back again. Identifying new customers creates value and growth opportunities.

You have a .csv summarizing the data. There are several variables: Customer ID, Book length in mins_avg (average of all purchases), Book length in minutes_sum (sum of all purchases), Price Paid_avg (average of all purchases), Price paid_sum (sum of all purchases), Review (a Boolean variable), Review (out of 10), Total minutes listened, Completion (from 0 to 1), Support requests (number), and Last visited minus purchase date (in days).

So these are the inputs (excluding customer ID, as it is completely arbitrary. It's more like a name, than a number).

The targets are a Boolean variable (so 0, or 1). We are taking a period of 2 years in our inputs, and the next 6 months as targets. So, in fact, we are predicting if: based on the last 2 years of activity and engagement, a customer will convert in the next 6 months. 6 months sounds like a reasonable time. If they don't convert after 6 months, chances are they've gone to a competitor or didn't like the Audiobook way of digesting information.

The task is simple: create a machine learning algorithm, which is able to predict if a customer will buy again.

This is a classification problem with two classes: won't buy and will buy, represented by 0s and 1s.

Good luck!

## Create a Machine Learning algorithm

In [24]:
import tensorflow as tf

## Data

In [25]:
#train set
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(float)
train_targets = npz['targets'].astype(int)

#validation set
npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(float)
validation_targets = npz['targets'].astype(int)

#test set
npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(float)
test_targets = npz['targets'].astype(int)

## Model

Outline, optimizers, loss, early stopping and training 

In [26]:
# Number of input features (columns) fed into the model
input_size = 10

# Number of output classes — 2, since this is binary classification
# (softmax with 2 units outputs a probability distribution over both classes)
output_size = 2

# Number of neurons in each hidden layer
hidden_layer_size = 50

# Build a simple feedforward (fully connected) neural network
model = tf.keras.Sequential([
    # First hidden layer: 50 neurons, ReLU activation
    # (ReLU introduces non-linearity and helps avoid vanishing gradients)
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),

    # Second hidden layer: another 50 neurons with ReLU activation
    # (adds more capacity for the model to learn complex patterns)
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),

    # Output layer: 2 neurons (one per class) with softmax activation,
    # which converts raw outputs into probabilities that sum to 1
    tf.keras.layers.Dense(output_size, activation='softmax')
])


# Configure the model for training:
# - optimizer='adam': Adam optimizer, an adaptive learning-rate method that
#   generally converges fast and works well without much tuning
# - loss='sparse_categorical_crossentropy': appropriate loss for multi-class
#   classification when targets are given as integer labels (e.g., 0 or 1)
#   rather than one-hot encoded vectors
# - metrics=['accuracy']: track classification accuracy during training/validation
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

batch_size = 100
max_epoch = 100

# Create an EarlyStopping callback to stop training automatically once
# the model stops improving on the validation set — this prevents
# overfitting and saves time instead of always running the full max_epoch
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)
# Train the model
model.fit(
    train_inputs,
    train_targets,
    batch_size=batch_size,
    epochs=max_epoch,
    callbacks=[early_stopping],
    validation_data=(validation_inputs,validation_targets),
    verbose=2  # print one line of output per epoch (concise logging)
)

Epoch 1/100
36/36 - 1s - 38ms/step - accuracy: 0.6759 - loss: 0.5830 - val_accuracy: 0.7226 - val_loss: 0.5070
Epoch 2/100
36/36 - 0s - 5ms/step - accuracy: 0.7578 - loss: 0.4706 - val_accuracy: 0.7606 - val_loss: 0.4332
Epoch 3/100
36/36 - 0s - 5ms/step - accuracy: 0.7804 - loss: 0.4183 - val_accuracy: 0.7740 - val_loss: 0.3939
Epoch 4/100
36/36 - 0s - 5ms/step - accuracy: 0.7969 - loss: 0.3904 - val_accuracy: 0.7785 - val_loss: 0.3737
Epoch 5/100
36/36 - 0s - 5ms/step - accuracy: 0.7994 - loss: 0.3745 - val_accuracy: 0.7987 - val_loss: 0.3741
Epoch 6/100
36/36 - 0s - 9ms/step - accuracy: 0.7997 - loss: 0.3670 - val_accuracy: 0.7987 - val_loss: 0.3608
Epoch 7/100
36/36 - 0s - 9ms/step - accuracy: 0.8036 - loss: 0.3590 - val_accuracy: 0.7919 - val_loss: 0.3673
Epoch 8/100
36/36 - 0s - 5ms/step - accuracy: 0.8075 - loss: 0.3510 - val_accuracy: 0.8098 - val_loss: 0.3544
Epoch 9/100
36/36 - 0s - 5ms/step - accuracy: 0.8100 - loss: 0.3460 - val_accuracy: 0.8054 - val_loss: 0.3478
Epoch 10/

## Test the model

In [27]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8638 - loss: 0.2951


In [28]:
print('\nTest loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))


Test loss: 0.30. Test accuracy: 86.38%
